# 03 · Getting real data with yfinance 🌐

Time to download **real stock-market data**. The `yfinance` library fetches price
history from Yahoo Finance and hands it back as a pandas DataFrame — the exact
table type you just practised on.

> ⚠️ **This notebook needs the internet.** If a download fails, check your
> connection and try again. Yahoo occasionally rate-limits — if so, wait a moment
> and re-run the cell.

## What you'll learn
- `import yfinance as yf`
- The **canonical download** we'll use everywhere (and why its options matter)
- Inspecting the result with `.head()`, `.shape`, `.columns`
- What each column (`Open`, `High`, `Low`, `Close`, `Volume`) means
- Trying a different ticker and time period
- Saving your data to a CSV file
- One company vs many — **multi-level columns**
- Building a **screener summary** — one row per stock

## 1. Import yfinance

In [ ]:
import yfinance as yf

## 2. Download some data (the form we'll always use)

`yf.download(...)` returns a DataFrame with **one row per trading day**. We pass a
few options every time so the table comes back tidy and easy to work with. Let's
grab six months of Apple.

In [ ]:
data = yf.download("AAPL", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
data.head()

### What those options mean

We'll use this **exact form** for every single-company download from now on. Here's
what each option does, in plain English:

- **`auto_adjust=True`** — tidy the prices for stock splits and dividends so the
  history lines up cleanly. You get one ready-to-use `Close` column (no separate
  "Adj Close").
- **`multi_level_index=False`** — give us **plain column names** (`Close`, `Open`,
  ...). This matters a lot: it means `data["Close"]` is a **single column** (a
  Series), which is exactly what our filtering and charts expect. Without it,
  yfinance stacks the ticker name on top of each column and `data["Close"]` comes
  back as a mini-table instead — confusing for now.
- **`progress=False`** — don't print a download progress bar; keeps the output
  clean.

Common periods you can pass: `"1mo"`, `"6mo"`, `"1y"`, `"5y"`, `"max"`.

## 3. Inspect it

Same tools as notebook 02 — because it's the same kind of object, a DataFrame.

In [ ]:
print("Shape (rows, cols):", data.shape)     # ~126 rows for 6 months
print("Columns:", list(data.columns))

In [ ]:
data.tail()          # the most recent few days

## 4. What the columns mean

| Column | Meaning |
| ------ | ------- |
| `Open` | Price when the market opened that day |
| `High` | Highest price during the day |
| `Low` | Lowest price during the day |
| `Close` | Price when the market closed — the "headline" price |
| `Volume` | How many shares changed hands that day |

The **date** is the row label (the **index**), not a normal column. We mostly
care about `Close`. Because we passed `multi_level_index=False`, `data["Close"]`
is a single column (a Series) you can compare and chart directly.

In [ ]:
data["Close"].tail()       # just the closing prices, most recent first at the end

In [ ]:
print("Highest close in the period:", data["Close"].max())
print("Lowest close in the period: ", data["Close"].min())
print("Average close:              ", data["Close"].mean())

### 🏋️ Your turn

Download **6 months of Microsoft** (`"MSFT"`) into a variable called `msft` — use
the **same options** as above — then show its `.head()`.

In [ ]:
# TODO: download MSFT for period "6mo" with auto_adjust=True, multi_level_index=False,
# progress=False into `msft`, then call msft.head().

**One way to do it:**

```python
msft = yf.download("MSFT", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
msft.head()
```

## 5. Try a different period

The `period` argument changes how far back you go. Here's a full year of Tesla —
same options, just a longer period.

In [ ]:
tsla = yf.download("TSLA", period="1y", auto_adjust=True, multi_level_index=False, progress=False)
print(tsla.shape)          # a year -> ~250 trading days
tsla.head()

### 🏋️ Your turn

Download just **1 month** (`"1mo"`) of Apple and print how many rows you got with
`.shape`. (Roughly 21 trading days in a month — weekends and holidays don't count.)

In [ ]:
# TODO: download AAPL for period "1mo" (same options) and print its .shape.

**One way to do it:**

```python
aapl_1mo = yf.download("AAPL", period="1mo", auto_adjust=True, multi_level_index=False, progress=False)
print(aapl_1mo.shape)
```

## 6. Save it to a file

Downloading every time is slow and needs internet. Save a DataFrame to a **CSV**
(a simple spreadsheet file) so you can reload it instantly later. The project has
a `data/` folder ready for this.

In [ ]:
data.to_csv("../data/aapl.csv")
print("Saved! Look in the data/ folder for aapl.csv")

To read it back later you'd use `pd.read_csv("../data/aapl.csv", index_col=0,
parse_dates=True)` — `index_col=0` keeps the date as the index, and
`parse_dates=True` reads it back as real dates.

## 7. One company vs many (multi-level columns) 🪜

So far we've asked for **one** company and used `multi_level_index=False` to get
flat column names — so `data["Close"]` is a single column (a Series). Start there:
it's the simple case, and it's all you need for the drill-down chart.

But a **screener** looks at *many* companies at once. Pass a **list** of tickers.
Now yfinance can't use plain column names — `Close` alone would be ambiguous
(whose close?) — so it gives you **two-level** columns like `('Close', 'AAPL')`:
the measurement on top, the ticker underneath.

In [ ]:
many = yf.download(["AAPL", "MSFT"], period="6mo", auto_adjust=True, progress=False)
many.head()

See how the column headers now have **two rows**? The top level is the measurement
(`Close`, `Open`, ...) and the second level is the ticker (`AAPL`, `MSFT`).

You dig in one level at a time:

- `many["Close"]` → **all the closes**, one column per company.
- `many["Close"]["AAPL"]` → **just Apple's** closes (back to a single Series).

In [ ]:
many["Close"].head()          # every company's closing prices, side by side

In [ ]:
many["Close"]["AAPL"].head()  # drill down to just Apple's closes -> a Series

### 🏋️ Your turn

From `many`, pull out **just Microsoft's** closing prices and show the first few.

In [ ]:
# TODO: from `many`, select MSFT's closes (Close, then MSFT) and call .head().

**One way to do it:**

```python
many["Close"]["MSFT"].head()
```

> 🧭 **Where this is heading:** those multi-level columns aren't just a curiosity —
> they're the raw material for a **stock screener**. A screener looks at *many*
> companies at once, **one row per company**, so you can filter and sort to find
> the interesting ones. Let's build that summary table now.

## 8. Build a screener summary table 📊

A screener flips the table around. Up to now every table has been **one row per
day** for a *single* company. A screener is **one row per company** — a scorecard
across a whole watchlist. We build it from a single multi-ticker download.

We bring in **pandas** (as `pd`) to assemble the new table.

In [ ]:
import pandas as pd

tickers = {"AAPL": "Apple", "MSFT": "Microsoft", "TSLA": "Tesla", "NVDA": "Nvidia", "KO": "Coca-Cola"}
raw = yf.download(list(tickers), period="6mo", auto_adjust=True, progress=False)
close = raw["Close"]            # one column per stock
summary = pd.DataFrame({
    "Name": pd.Series(tickers),
    "Price": close.iloc[-1],
    "Change %": (close.iloc[-1] / close.iloc[0] - 1) * 100,
    "Avg Volume": raw["Volume"].mean(),
})
summary.round(2)

### Reading that, line by line

- **`tickers = {...}`** — a little lookup: each ticker symbol → a friendly name.
- **`yf.download(list(tickers), ...)`** — `list(tickers)` is just the symbols
  (`["AAPL", "MSFT", ...]`); one call fetches all five at once. We **don't** pass
  `multi_level_index=False` here, so we get the **multi-level** columns from
  section 7.
- **`close = raw["Close"]`** — pull out just the `Close` block: one column per
  company, one row per day.
- **`pd.DataFrame({...})`** — build a **brand-new** table, **one row per company**:
  - `"Name"` — `pd.Series(tickers)` turns the lookup into a column of names.
  - `"Price"` — `close.iloc[-1]` is the **last row** (most recent day), i.e. each
    company's latest price.
  - `"Change %"` — last price ÷ first price − 1, ×100: the percent move over the
    whole period.
  - `"Avg Volume"` — `raw["Volume"].mean()` averages each company's volume column.
- **`.round(2)`** — tidy the numbers to 2 decimal places.

Every column lines up because they're all indexed by the **ticker**. That single
table — one row per stock — **is** the screener.

### 🏋️ Your turn

Add **a couple more companies** to the `tickers` dictionary (say
`"AMZN": "Amazon"` and `"GOOGL": "Alphabet"`), then re-run the cell above. Your new
stocks appear as **new rows** — no other code changes.

In [ ]:
# TODO: add two more entries to `tickers`, then re-run the download + summary cell.

**One way to do it:**

```python
tickers = {
    "AAPL": "Apple", "MSFT": "Microsoft", "TSLA": "Tesla", "NVDA": "Nvidia",
    "KO": "Coca-Cola", "AMZN": "Amazon", "GOOGL": "Alphabet",
}
# then re-run the `raw = yf.download(...)` / `summary = ...` cell above.
```

## 🎉 You're pulling live data!

You can now fetch real prices for any ticker and period, inspect the table, save it
to CSV — and, crucially, turn a multi-ticker download into a **screener summary**:
one row per company. That `summary` table is the backbone of the app you'll build.

**Next up:** [`04_filtering_and_searching.ipynb`](04_filtering_and_searching.ipynb)
— the heart of the project: **filtering and searching** that table to find the
interesting stocks.

📖 Companion guide:
[`../guides/03-getting-data-yfinance.md`](../guides/03-getting-data-yfinance.md)